In [1]:
import pandas as pd
import numpy as np 
from surprise import KNNBasic 
from surprise import Dataset 
from surprise.model_selection import cross_validate, train_test_split
from surprise import Dataset, NormalPredictor, Reader, SVD, accuracy
from collections import defaultdict


books = pd.read_csv('Books.csv', dtype={'Year-Of-Publication': str})
ratings = pd.read_csv('Ratings.csv')
users = pd.read_csv('Users.csv')

#muutetaan year-of.publication numeroksi, virheelliset NaN:iksi
books['Year-Of-Publication'] = pd.to_numeric(books['Year-Of-Publication'], errors='coerce')


In [2]:
# Pudota arvioinnit, joissa Book-Rating puuttuu
ratings = ratings.dropna(subset=['Book-Rating'])


ratings['User-ID'] = ratings['User-ID'].astype(str)
ratings['ISBN'] = ratings['ISBN'].astype(str)


reader = Reader(rating_scale=(0,10))
data = Dataset.load_from_df(ratings[['User-ID', 'ISBN', 'Book-Rating']], reader)

Koulutetaan malli


In [3]:
trainset, testset = train_test_split(data, test_size=0.25, random_state=42)
algo=SVD(random_state=42)
algo.fit(trainset)
predictions = algo.test(testset)
print("RMSE: ", accuracy.rmse(predictions))
print("MAE:", accuracy.mae(predictions))

RMSE: 3.4984
RMSE:  3.4984200886927654
MAE:  2.8167
MAE: 2.8166896127862153


In [4]:
def get_top_n(predictions, n=10):
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid,est))
    #järjestetään ja otetaan top-N
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

top_n = get_top_n(predictions, n=10)

In [8]:

def recommend_for_all_users(algo, ratings_df, books=None, n=10):
    all_users = ratings_df['User-ID'].unique()
    all_items = ratings_df['ISBN'].unique()
    results = []

    for user_id in all_users:
        rated_items = set(ratings_df[ratings_df['User-ID'] == user_id]['ISBN'].unique())
        candidates = [iid for iid in all_items if iid not in rated_items]

        preds = [algo.predict(str(user_id), iid) for iid in candidates]
        preds_sorted = sorted(preds, key=lambda p: p.est, reverse=True)[:n]

        for p in preds_sorted:
            results.append((p.uid, p.iid, p.est))

    df = pd.DataFrame(results, columns=['User-ID', 'ISBN', 'Estimated-rating'])

    if books is not None and 'ISBN' in books.columns:
        df = df.merge(
            books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher']],
            on='ISBN',
            how='left'
        )
    return df


topn_all = recommend_for_all_users(algo, ratings, books=books, n=10)
topn_all.head()

topn_all.to_csv('recommendations.csv', index=False)


KeyboardInterrupt: 